In [1]:
import os
os.chdir('/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning')
print("Current working directory: {0}".format(os.getcwd()))

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload
%autoreload 2

Current working directory: /Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning


In [2]:
import math
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
import os
from dataclasses import dataclass
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from tqdm import tqdm

import torch
from torch import nn as nn
from torch.nn import functional as F

from batchbald_redux import (
    active_learning,
    batchbald,
    consistent_mc_dropout,
    joint_entropy,
    repeated_mnist,
)


In [3]:
use_cuda = torch.cuda.is_available()
print(f"use_cuda: {use_cuda}")
device = "cuda" if use_cuda else "cpu"
kwargs = {"num_workers": 0, "pin_memory": True} if use_cuda else {}

use_cuda: False


In [4]:
class BayesianCNN(consistent_mc_dropout.BayesianModule):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=5)
        self.conv1_drop = consistent_mc_dropout.ConsistentMCDropout2d()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        self.conv2_drop = consistent_mc_dropout.ConsistentMCDropout2d()
        self.fc1 = nn.Linear(1024, 128)
        self.fc1_drop = consistent_mc_dropout.ConsistentMCDropout()
        self.fc2 = nn.Linear(128, num_classes)

    def mc_forward_impl(self, input: torch.Tensor):
        input = F.relu(F.max_pool2d(self.conv1_drop(self.conv1(input)), 2))
        input = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(input)), 2))
        input = input.view(-1, 1024)
        input = F.relu(self.fc1_drop(self.fc1(input)))
        input = self.fc2(input)
        input = F.log_softmax(input, dim=1)

        return input

In [6]:
algo_list = ["batchbald"]
final_test_accs = []
final_indices = []

max_training_samples = 180  # Maximum limit of train samples needed
num_inference_samples = 10
num_test_inference_samples = 5
num_samples = 100000  # Total number of samples

test_batch_size = 512  # Test Loader Batch size
batch_size = 64  # Train loader Batch size
scoring_batch_size = 128  # Pool Loader Batch size
training_iterations = 4096 * 6

acquisition_batch_size = 4

In [7]:
seed_value = 0

torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.backends.cudnn.deterministic = True
torch.cuda.manual_seed_all(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
os.environ["PYTHONHASHSEED"] = str(seed_value)

num_initial_samples = 20  # Number of initial samples required
num_classes = 10  # Total classes in MNIST dataset

# get an active learning dataset
train_dataset = MNIST(root='./data/raw', train=True, transform=transforms.ToTensor())
test_dataset = MNIST(root='./data/raw', train=False, transform=transforms.ToTensor())

# Generates 20 samples (2 from each class) and returns their indices
initial_samples = active_learning.get_balanced_sample_indices(
    repeated_mnist.get_targets(train_dataset),
    num_classes=num_classes,
    n_per_digit=num_initial_samples / num_classes,
)

test_accs = []
test_loss = []
added_indices = []

active_learning_data = active_learning.ActiveLearningData(
    train_dataset
)  # Splits the dataset into training dataset and pool dataset

active_learning_data.acquire(
    initial_samples
)  # Seperates the initial indices from the pool and fixes it as initial train dataset

train_loader = torch.utils.data.DataLoader(
    active_learning_data.training_dataset,
    sampler=active_learning.RandomFixedLengthSampler(
        active_learning_data.training_dataset, training_iterations
    ),
    batch_size=batch_size,
    **kwargs,
)

pool_loader = torch.utils.data.DataLoader(
    active_learning_data.pool_dataset,
    batch_size=scoring_batch_size,
    shuffle=False,
    **kwargs,
)

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, **kwargs)

pbar = tqdm(
    initial=len(active_learning_data.training_dataset),
    total=max_training_samples,
    desc="Training Set Size",
)

while True:
    model = BayesianCNN(num_classes).to(device=device)  # initialise model
    optimizer = torch.optim.Adam(model.parameters())

    model.train()

    # Train
    for data, target in tqdm(train_loader, desc="Training", leave=False):
        data = data.to(device=device)
        target = target.to(device=device)

        optimizer.zero_grad()

        prediction = model(data, 1).squeeze(1)
        loss = F.nll_loss(prediction, target)

        loss.backward()
        optimizer.step()

    # Test
    loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in tqdm(test_loader, desc="Testing", leave=False):
            data = data.to(device=device)
            target = target.to(device=device)

            prediction = torch.logsumexp(model(data, num_test_inference_samples), dim=1) - math.log(
                num_test_inference_samples
            )
            loss += F.nll_loss(prediction, target, reduction="sum")

            prediction = prediction.max(1)[1]
            correct += prediction.eq(target.view_as(prediction)).sum().item()

    loss /= len(test_loader.dataset)
    test_loss.append(loss)

    percentage_correct = 100.0 * correct / len(test_loader.dataset)
    test_accs.append(percentage_correct)

    print("Test set: Average loss: {:.4f}, Accuracy: ({:.2f}%)".format(loss, percentage_correct))

    if len(active_learning_data.training_dataset) >= max_training_samples:
        break

    # Acquire pool predictions
    N = len(active_learning_data.pool_dataset)
    logits_N_K_C = torch.empty(
        (N, num_inference_samples, num_classes),
        dtype=torch.double,
        pin_memory=use_cuda,
    )

    with torch.no_grad():
        model.eval()

        for i, (data, _) in enumerate(tqdm(pool_loader, desc="Evaluating Acquisition Set", leave=False)):
            data = data.to(device=device)

            lower = i * pool_loader.batch_size
            upper = min(lower + pool_loader.batch_size, N)
            logits_N_K_C[lower:upper].copy_(model(data, num_inference_samples).double(), non_blocking=True)

    with torch.no_grad():
        candidate_batch = batchbald.get_batchbald_batch(
            logits_N_K_C,
            acquisition_batch_size,
            num_samples,
            dtype=torch.double,
            device=device,  # Returns the indices and scores(Mutual Information) for the batch selected by Batchbald/BALD Strategy.
        )

    targets = repeated_mnist.get_targets(active_learning_data.pool_dataset)  # Returns the target labels
    dataset_indices = active_learning_data.get_dataset_indices(
        candidate_batch.indices
    )  # Returns indices for candidate batch

    print("Dataset indices: ", dataset_indices)
    # print("Scores: ", candidate_batch.scores)
    print("Labels: ", targets[candidate_batch.indices])

    active_learning_data.acquire(candidate_batch.indices)  # add the new indices to training dataset
    added_indices.append(dataset_indices)
    pbar.update(len(dataset_indices))
final_test_accs.append(test_accs)
final_indices.append(added_indices)

Training Set Size:  11%|█         | 20/180 [00:00<?, ?it/s]

In [10]:
import pickle

with open("/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning/test_dict", 'rb') as f:
    x = pickle.load(f)

In [11]:
x

{'acc': [60.65, 61.39, 65.55, 63.13],
 'loss': [array(1.8204486, dtype=float32),
  array(2.3114674, dtype=float32),
  array(2.0355117, dtype=float32),
  array(2.46819, dtype=float32)],
 'dataset_len': 32}

In [9]:
pwd

'/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning/notebooks'